In [2]:
#!/usr/bin/env python3
"""
make_figure5.py

Replot manuscript Figure 5 from saved plot-ready data.

Required input in the running directory:

    figure5_plotdata.parquet

Outputs:

    figure5.pdf
    figure5.png
    figure5.svg
"""

from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib import cm, colors
from matplotlib.ticker import MaxNLocator


# ============================================================
# Paths
# ============================================================

try:
    RUN_DIR = Path(__file__).resolve().parent
except NameError:
    RUN_DIR = Path.cwd()

PLOTDATA_FILE = RUN_DIR / "figure5_plotdata.parquet"

OUT_PDF = RUN_DIR / "figure5.pdf"
OUT_PNG = RUN_DIR / "figure5.png"
OUT_SVG = RUN_DIR / "figure5.svg"


# ============================================================
# Constants
# ============================================================

CMAP_EMP = cm.Greys
CMAP_DIFF = cm.PuOr_r

NORM_EMP = colors.Normalize(vmin=0.0, vmax=1.0)
NORM_DIFF = colors.TwoSlopeNorm(vmin=-0.50, vcenter=0.0, vmax=0.50)

GLOBAL_COLOR = "#2f789b"
AGE_COLOR = "#5aa052"

# The saved D_JS values are in nats; the manuscript reports bits.
DJS_VALUES_ARE_NATS = True


# ============================================================
# Style
# ============================================================

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],

    "font.size": 7.2,
    "axes.titlesize": 8.0,
    "axes.labelsize": 7.2,
    "xtick.labelsize": 6.2,
    "ytick.labelsize": 6.2,
    "legend.fontsize": 5.6,

    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": False,
    "axes.linewidth": 0.45,

    "xtick.major.width": 0.35,
    "ytick.major.width": 0.35,
    "xtick.major.size": 2.0,
    "ytick.major.size": 2.0,

    "legend.frameon": False,

    "figure.dpi": 300,
    "savefig.dpi": 600,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})


# ============================================================
# Data helpers
# ============================================================

def fmt_count(n: float) -> str:
    n = float(n)

    if n >= 1_000_000:
        x = n / 1_000_000
        return f"{x:.1f}M" if x < 10 else f"{x:.0f}M"

    if n >= 1_000:
        x = n / 1_000
        return f"{x:.1f}k" if x < 10 else f"{x:.0f}k"

    return f"{n:.0f}"


def math_row_label(label: str) -> str:
    label = str(label)

    if label.startswith("c="):
        return r"$c=" + label.replace("c=", "") + "$"

    return label


def js_to_bits(x: float) -> float:
    x = float(x)
    if DJS_VALUES_ARE_NATS:
        return x / np.log(2.0)
    return x


def get_composition(
    plotdata: pd.DataFrame,
    layer_key: str,
    representation: str,
) -> pd.DataFrame:
    sub = plotdata[
        (plotdata["row_type"].astype(str) == "composition")
        & (plotdata["layer_key"].astype(str) == layer_key)
        & (plotdata["representation"].astype(str) == representation)
    ].copy()

    if sub.empty:
        raise ValueError(
            f"No composition data found for layer_key={layer_key}, "
            f"representation={representation}"
        )

    return sub


def get_js(
    plotdata: pd.DataFrame,
    representation: str,
) -> pd.DataFrame:
    sub = plotdata[
        (plotdata["row_type"].astype(str) == "js_bar")
        & (plotdata["representation"].astype(str) == representation)
    ].copy()

    if sub.empty:
        raise ValueError(f"No JSD data found for representation={representation}")

    return sub


# ============================================================
# Heatmap primitives
# ============================================================

def draw_counts_axis(
    ax,
    y_positions: np.ndarray,
    counts: list[float],
    nr: int,
    bar_h: float,
    show_xlabel: bool = True,
) -> None:
    cmax = max(counts) if counts else 1.0

    for y, n in zip(y_positions, counts):
        ax.barh(
            y,
            n,
            height=bar_h * 0.60,
            color="#f2f2f2",
            edgecolor="#333333",
            linewidth=0.55,
            zorder=2,
        )

    ax.set_xlim(0, cmax * 1.10)
    ax.invert_xaxis()
    ax.set_ylim(-0.72, nr - 0.28)

    ax.set_yticks([])
    ax.set_xticks([0, cmax])
    ax.set_xticklabels(["0", fmt_count(cmax)])

    if show_xlabel:
        ax.set_xlabel("# groups", labelpad=2.0)
    else:
        ax.set_xlabel("")

    ax.spines["left"].set_visible(False)
    ax.spines["right"].set_visible(True)
    ax.spines["right"].set_linewidth(0.40)
    ax.spines["top"].set_visible(False)

    ax.tick_params(axis="x", length=2.0, pad=2.0)


def draw_label_axis(
    ax,
    y_positions: np.ndarray,
    row_labels: list[str],
    nr: int,
) -> None:
    ax.set_xlim(0, 1)
    ax.set_ylim(-0.72, nr - 0.28)
    ax.axis("off")

    for y, lab in zip(y_positions, row_labels):
        ax.text(
            0.50,
            y,
            math_row_label(lab),
            rotation=82,
            ha="center",
            va="center",
            fontsize=4.6,
            fontweight="bold",
        )


def draw_heat_rows(
    ax,
    sub: pd.DataFrame,
    row_orders: list[int],
    y_by_row: dict[int, float],
    value_col: str,
    cmap,
    norm,
    bar_h: float,
    annotate_values: bool = False,
) -> None:
    for row_order in row_orders:
        row = sub[sub["row_order"].astype(int) == int(row_order)].sort_values("x_left")
        y = y_by_row[int(row_order)]

        # Outer border only. No internal white segment separators.
        ax.add_patch(
            Rectangle(
                (0.0, y - bar_h / 2),
                1.0,
                bar_h,
                facecolor="none",
                edgecolor="#333333",
                linewidth=0.38,
                zorder=5,
                clip_on=False,
            )
        )

        nseg = len(row)

        # All internal values are vertical, as in the dense manuscript panels.
        if nseg <= 6:
            text_fs = 3.05
        elif nseg <= 11:
            text_fs = 2.35
        else:
            text_fs = 1.85

        for _, r in row.iterrows():
            x0 = float(r["x_left"])
            x1 = float(r["x_right"])

            # Guard against clipping at exactly 0 or 1.
            x0 = max(0.0, x0)
            x1 = min(1.0, x1)
            width = max(0.0, x1 - x0)

            value = r[value_col]
            value = 0.0 if pd.isna(value) else float(value)

            ax.add_patch(
                Rectangle(
                    (x0, y - bar_h / 2),
                    width,
                    bar_h,
                    facecolor=cmap(norm(value)),
                    edgecolor="none",
                    linewidth=0.0,
                    zorder=2,
                    clip_on=False,
                )
            )

            if annotate_values:
                raw = r["real"]
                raw = 0.0 if pd.isna(raw) else float(raw)

                scaled = r["real_row_scaled"]
                scaled = 0.0 if pd.isna(scaled) else float(scaled)

                text_color = "white" if scaled > 0.58 else "#222222"

                ax.text(
                    x0 + width / 2,
                    y,
                    f"{raw:.2f}",
                    ha="center",
                    va="center",
                    fontsize=text_fs,
                    rotation=90,
                    color=text_color,
                    zorder=6,
                    clip_on=True,
                )


def finish_heat_axis(
    ax,
    nr: int,
    show_xticklabels: bool,
    title: str | None = None,
) -> None:
    # Slightly padded x-limits prevent the right edge from looking cut.
    ax.set_xlim(-0.004, 1.004)
    ax.set_ylim(-0.72, nr - 0.28)
    ax.set_yticks([])
    ax.set_xticks([0, 0.5, 1.0])

    if show_xticklabels:
        ax.set_xticklabels(["0", "50", "100"])
    else:
        ax.set_xticklabels([])

    for side in ["top", "right", "left"]:
        ax.spines[side].set_visible(False)

    ax.spines["bottom"].set_linewidth(0.35)
    ax.tick_params(axis="x", length=2.0, pad=2.0)

    if title is not None:
        ax.set_title(title, fontsize=7.1, pad=4.2)


def draw_shared_fraction_xlabel(
    fig,
    x_emp: float,
    w_emp: float,
    x_diff: float,
    w_diff: float,
    bottom: float,
    dy: float,
) -> None:
    x_center = (x_emp + w_emp / 2 + x_diff + w_diff / 2) / 2

    fig.text(
        x_center,
        bottom - dy,
        "Vaccinated fraction (%)",
        ha="center",
        va="top",
        fontsize=7.0,
    )


def draw_heatmap_block(
    fig,
    rect: list[float],
    plotdata: pd.DataFrame,
    layer_key: str,
    representation: str,
    panel_label: str,
    layer_title: str | None,
    show_xlabel: bool,
    show_column_titles: bool,
    annotate_values: bool,
    show_counts_xlabel: bool = True,
    xlabel_dy: float = 0.045,
) -> None:
    """
    Draw one manuscript-style block.

    rect = [left, bottom, width, height] in figure coordinates.

    Internal layout:
        counts | row labels | empirical | gap | deviation

    Only one shared x-label is drawn below Empirical + Deviation.
    """
    sub = get_composition(plotdata, layer_key, representation)

    row_info = (
        sub.groupby("row_order", as_index=False)
        .agg(
            row_label=("row_label", "first"),
            n_groups=("n_groups", "max"),
        )
        .sort_values("row_order")
        .reset_index(drop=True)
    )

    row_orders = row_info["row_order"].astype(int).tolist()
    row_labels = row_info["row_label"].astype(str).tolist()
    counts = row_info["n_groups"].astype(float).tolist()

    nr = len(row_orders)
    y_positions = np.arange(nr)[::-1]
    y_by_row = dict(zip(row_orders, y_positions))

    left, bottom, width, height = rect

    # Internal proportions tuned to manuscript target.
    x_counts = left
    w_counts = 0.195 * width

    x_labels = left + 0.224 * width
    w_labels = 0.050 * width

    x_emp = left + 0.322 * width
    w_emp = 0.302 * width

    x_diff = left + 0.668 * width
    w_diff = 0.302 * width

    row_h = 0.58

    ax_counts = fig.add_axes([x_counts, bottom, w_counts, height])
    ax_labels = fig.add_axes([x_labels, bottom, w_labels, height])
    ax_emp = fig.add_axes([x_emp, bottom, w_emp, height])
    ax_diff = fig.add_axes([x_diff, bottom, w_diff, height])

    draw_counts_axis(
        ax_counts,
        y_positions=y_positions,
        counts=counts,
        nr=nr,
        bar_h=row_h,
        show_xlabel=show_counts_xlabel,
    )

    draw_label_axis(
        ax_labels,
        y_positions=y_positions,
        row_labels=row_labels,
        nr=nr,
    )

    draw_heat_rows(
        ax_emp,
        sub=sub,
        row_orders=row_orders,
        y_by_row=y_by_row,
        value_col="real_row_scaled",
        cmap=CMAP_EMP,
        norm=NORM_EMP,
        bar_h=row_h,
        annotate_values=annotate_values,
    )

    draw_heat_rows(
        ax_diff,
        sub=sub,
        row_orders=row_orders,
        y_by_row=y_by_row,
        value_col="diff",
        cmap=CMAP_DIFF,
        norm=NORM_DIFF,
        bar_h=row_h,
        annotate_values=False,
    )

    finish_heat_axis(
        ax_emp,
        nr=nr,
        show_xticklabels=show_xlabel,
        title="Empirical" if show_column_titles else None,
    )

    finish_heat_axis(
        ax_diff,
        nr=nr,
        show_xticklabels=show_xlabel,
        title="Deviation from null" if show_column_titles else None,
    )

    if show_xlabel:
        draw_shared_fraction_xlabel(
            fig,
            x_emp=x_emp,
            w_emp=w_emp,
            x_diff=x_diff,
            w_diff=w_diff,
            bottom=bottom,
            dy=xlabel_dy,
        )

    ax_counts.text(
        -0.12,
        1.085,
        panel_label,
        transform=ax_counts.transAxes,
        ha="left",
        va="bottom",
        fontsize=9.3,
        fontweight="bold",
        clip_on=False,
    )

    if layer_title:
        ax_emp.text(
            0.50,
            1.18,
            layer_title,
            transform=ax_emp.transAxes,
            ha="center",
            va="bottom",
            fontsize=9.5,
            fontweight="bold",
            clip_on=False,
        )


# ============================================================
# Colour bars
# ============================================================

def draw_colourbars(fig, e_bottom: float, e_height: float, js_gap: float) -> None:
    """
    Colour bars are taller and moved left so their ticks do not collide with F/G.
    """
    js_h = (e_height - js_gap) / 2.0
    g_y = e_bottom
    f_y = e_bottom + js_h + js_gap

    cb_x = 0.660
    cb_w = 0.016
    cb_margin = 0.000

    ax_emp = fig.add_axes([cb_x, f_y + cb_margin, cb_w, js_h - 2 * cb_margin])
    ax_diff = fig.add_axes([cb_x, g_y + cb_margin, cb_w, js_h - 2 * cb_margin])

    sm_emp = cm.ScalarMappable(norm=NORM_EMP, cmap=CMAP_EMP)
    sm_emp.set_array([])

    cb_emp = fig.colorbar(sm_emp, cax=ax_emp)
    cb_emp.outline.set_linewidth(0.30)
    cb_emp.set_ticks([])
    cb_emp.ax.tick_params(
        left=False,
        right=False,
        labelleft=False,
        labelright=False,
    )

    ax_emp.text(
        0.50,
        1.20,
        "Empirical",
        transform=ax_emp.transAxes,
        ha="center",
        va="bottom",
        fontsize=5.8,
        clip_on=False,
    )

    ax_emp.text(
        0.50,
        1.04,
        "max",
        transform=ax_emp.transAxes,
        ha="center",
        va="bottom",
        fontsize=5.3,
        clip_on=False,
    )

    ax_emp.text(
        0.50,
        -0.04,
        "min",
        transform=ax_emp.transAxes,
        ha="center",
        va="top",
        fontsize=5.3,
        clip_on=False,
    )

    sm_diff = cm.ScalarMappable(norm=NORM_DIFF, cmap=CMAP_DIFF)
    sm_diff.set_array([])

    cb_diff = fig.colorbar(sm_diff, cax=ax_diff)
    cb_diff.outline.set_linewidth(0.30)
    cb_diff.set_ticks([-0.50, -0.25, 0.00, 0.25, 0.50])
    cb_diff.ax.tick_params(
        labelsize=5.2,
        width=0.30,
        length=1.3,
        pad=1.0,
        labelleft=False,
        labelright=True,
    )

    ax_diff.text(
        0.50,
        1.18,
        "Deviation\nfrom null",
        transform=ax_diff.transAxes,
        ha="center",
        va="bottom",
        fontsize=5.8,
        linespacing=0.95,
        clip_on=False,
    )


# ============================================================
# JSD panels
# ============================================================

def draw_js_panel(
    ax,
    plotdata: pd.DataFrame,
    representation: str,
    panel_label: str,
    title: str,
    show_legend: bool,
    show_xticklabels: bool,
) -> None:
    sub = get_js(plotdata, representation)

    layers = ["Work", "Education", "Household"]
    x = np.arange(len(layers))
    width = 0.32

    vals_global = []
    vals_age = []

    for layer in layers:
        s = sub[sub["layer"].astype(str) == layer]

        g = s[s["null_model"].astype(str) == "Global shuffle"]
        a = s[s["null_model"].astype(str) == "Age-stratified"]

        vals_global.append(js_to_bits(g["value"].iloc[0]) if not g.empty else np.nan)
        vals_age.append(js_to_bits(a["value"].iloc[0]) if not a.empty else np.nan)

    bars_global = ax.bar(
        x - width / 2,
        vals_global,
        width=width,
        color=GLOBAL_COLOR,
        edgecolor="white",
        linewidth=0.35,
        label="Global shuffle",
    )

    bars_age = ax.bar(
        x + width / 2,
        vals_age,
        width=width,
        color=AGE_COLOR,
        edgecolor="white",
        linewidth=0.35,
        label="Age-stratified",
    )

    for bars in [bars_global, bars_age]:
        for b in bars:
            h = b.get_height()
            if np.isfinite(h):
                ax.text(
                    b.get_x() + b.get_width() / 2,
                    h + 0.012,
                    f"{h:.2f}",
                    ha="center",
                    va="bottom",
                    fontsize=5.2,
                )

    ax.set_ylim(0, 0.65)
    ax.set_ylabel(r"$D_{JS}$ (bits)", labelpad=1.6)

    ax.set_title(
        title,
        fontsize=7.1,
        fontweight="bold",
        pad=2.5,
    )

    ax.set_xticks(x)

    if show_xticklabels:
        ax.set_xticklabels(layers, rotation=27, ha="right")
    else:
        ax.set_xticklabels([])

    ax.yaxis.set_major_locator(MaxNLocator(4))
    ax.tick_params(axis="both", labelsize=5.6, pad=1.3)

    # F and G are boxed in the manuscript.
    for side in ["top", "right", "left", "bottom"]:
        ax.spines[side].set_visible(True)
        ax.spines[side].set_linewidth(0.45)
        ax.spines[side].set_color("#333333")

    ax.text(
        -0.13,
        1.06,
        panel_label,
        transform=ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=8.4,
        fontweight="bold",
        clip_on=False,
    )

    if show_legend:
        ax.legend(
            loc="upper left",
            bbox_to_anchor=(0.02, 0.98),
            frameon=False,
            fontsize=5.1,
            handlelength=0.90,
            handletextpad=0.28,
            labelspacing=0.10,
            borderaxespad=0.0,
        )


# ============================================================
# Main figure
# ============================================================

def make_figure5(plotdata: pd.DataFrame) -> None:
    fig = plt.figure(figsize=(7.80, 5.03))

    # --------------------------------------------------------
    # A-C: small cliques
    # --------------------------------------------------------

    draw_heatmap_block(
        fig,
        rect=[0.030, 0.545, 0.300, 0.395],
        plotdata=plotdata,
        layer_key="work",
        representation="small",
        panel_label="A.",
        layer_title="Work",
        show_xlabel=False,
        show_column_titles=True,
        annotate_values=True,
        show_counts_xlabel=False,
    )

    draw_heatmap_block(
        fig,
        rect=[0.345, 0.545, 0.300, 0.395],
        plotdata=plotdata,
        layer_key="school_new",
        representation="small",
        panel_label="B.",
        layer_title="Education",
        show_xlabel=False,
        show_column_titles=True,
        annotate_values=True,
        show_counts_xlabel=False,
    )

    draw_heatmap_block(
        fig,
        rect=[0.660, 0.545, 0.300, 0.395],
        plotdata=plotdata,
        layer_key="house",
        representation="small",
        panel_label="C.",
        layer_title="Household",
        show_xlabel=True,
        show_column_titles=True,
        annotate_values=True,
        show_counts_xlabel=True,
        xlabel_dy=0.038,
    )

    # --------------------------------------------------------
    # D-E: large cliques
    # --------------------------------------------------------

    e_bottom = 0.085
    e_height = 0.340

    draw_heatmap_block(
        fig,
        rect=[0.030, e_bottom, 0.300, e_height],
        plotdata=plotdata,
        layer_key="work",
        representation="large",
        panel_label="D.",
        layer_title=None,
        show_xlabel=True,
        show_column_titles=False,
        annotate_values=False,
        show_counts_xlabel=True,
        xlabel_dy=0.052,
    )

    draw_heatmap_block(
        fig,
        rect=[0.345, e_bottom, 0.300, e_height],
        plotdata=plotdata,
        layer_key="school_new",
        representation="large",
        panel_label="E.",
        layer_title=None,
        show_xlabel=True,
        show_column_titles=False,
        annotate_values=False,
        show_counts_xlabel=True,
        xlabel_dy=0.052,
    )

    # --------------------------------------------------------
    # Colour bars and F/G
    # F + G combined vertical span equals panel E.
    # --------------------------------------------------------

    js_gap = 0.060
    js_h = (e_height - js_gap) / 2.0
    js_x = 0.760
    js_w = 0.190

    g_y = e_bottom
    f_y = e_bottom + js_h + js_gap

    draw_colourbars(fig, e_bottom=e_bottom, e_height=e_height, js_gap=js_gap)

    ax_f = fig.add_axes([js_x, f_y, js_w, js_h])
    draw_js_panel(
        ax_f,
        plotdata,
        representation="small",
        panel_label="F.",
        title="Small cliques",
        show_legend=True,
        show_xticklabels=False,
    )

    ax_g = fig.add_axes([js_x, g_y, js_w, js_h])
    draw_js_panel(
        ax_g,
        plotdata,
        representation="large",
        panel_label="G.",
        title=r"Large cliques ($c > 20$)",
        show_legend=False,
        show_xticklabels=True,
    )

    # --------------------------------------------------------
    # Row labels
    # --------------------------------------------------------

    fig.text(
        0.012,
        0.735,
        "Small cliques",
        rotation=90,
        ha="center",
        va="center",
        fontsize=8.4,
        fontweight="bold",
    )

    fig.text(
        0.012,
        0.285,
        "Large cliques",
        rotation=90,
        ha="center",
        va="center",
        fontsize=8.4,
        fontweight="bold",
    )

    fig.savefig(OUT_PDF)
    fig.savefig(OUT_PNG, dpi=600)
    fig.savefig(OUT_SVG)

    plt.close(fig)

    print("Saved:")
    print(f"  {OUT_PDF}")
    print(f"  {OUT_PNG}")
    print(f"  {OUT_SVG}")


# ============================================================
# Run
# ============================================================

def main() -> None:
    if not PLOTDATA_FILE.exists():
        raise FileNotFoundError(
            f"Could not find:\n  {PLOTDATA_FILE}\n\n"
            "Put figure5_plotdata.parquet in the same folder as this script/notebook."
        )

    print("Reading:")
    print(f"  {PLOTDATA_FILE}")

    plotdata = pd.read_parquet(PLOTDATA_FILE)

    required_cols = [
        "row_type",
        "representation",
        "layer_key",
        "layer",
        "row_order",
        "row_label",
        "x_left",
        "x_right",
        "n_groups",
        "real",
        "real_row_scaled",
        "diff",
        "null_model",
        "value",
    ]

    missing = [c for c in required_cols if c not in plotdata.columns]
    if missing:
        raise ValueError(
            "figure5_plotdata.parquet is missing required columns:\n"
            + "\n".join(f"  - {c}" for c in missing)
        )

    make_figure5(plotdata)


if __name__ == "__main__":
    main()

Reading:
  /home/karimia1/social_fabric_vacc/github/figure5_plotdata.parquet
Saved:
  /home/karimia1/social_fabric_vacc/github/figure5.pdf
  /home/karimia1/social_fabric_vacc/github/figure5.png
  /home/karimia1/social_fabric_vacc/github/figure5.svg
